# We will process a df to visualize all duals by team

In [15]:
import pandas as pd
import numpy as np
import re

In [16]:
# read in data
duals = pd.read_csv('dual_meets.xls', parse_dates=["event_date"]).sort_values(by='event_date')
display(duals)

duals.columns

,dual_id,team1_id,team1_rank,team1_score,team2_id,team2_rank,team2_score,event_date
279,280,24,72,19,67,76,18,2025-11-01
283,284,20,32,19,64,47,13,2025-11-01
282,283,70,36,19,23,11,12,2025-11-01
281,282,18,31,36,62,33,3,2025-11-01
280,281,27,19,20,68,21,12,2025-11-01
...,...,...,...,...,...,...,...,...
550,551,39,4,20,73,14,14,2026-02-22
549,550,52,2,32,17,5,11,2026-02-22
548,549,25,7,34,5,43,6,2026-02-22
552,553,1,25,16,18,22,15,2026-02-22


Index(['dual_id', 'team1_id', 'team1_rank', 'team1_score', 'team2_id',
       'team2_rank', 'team2_score', 'event_date'],
      dtype='object')

In [17]:
teams = pd.read_csv('teams.xls')
teams

,team_id,name
0,1,Northern Iowa
1,2,South Dakota State
2,3,Drexel
3,4,Northern Illinois
4,5,Central Michigan
...,...,...
73,74,Indiana
74,75,Army West Point
75,76,Wyoming
76,77,Bellarmine


In [18]:
# Create a mapping of team_id to team_name
team_name_map = dict(zip(teams['team_id'], teams['name']))

# Initialize empty list to store processed dual meets
team_history = []

# Process each dual meet
for idx, row in duals.iterrows():
    dual_id = row['dual_id']
    event_date = row['event_date']
    
    # Team 1 info
    team1_id = row['team1_id']
    team1_name = team_name_map.get(team1_id, f"Unknown Team {team1_id}")
    team1_score = row['team1_score']
    team1_rank = row['team1_rank']
    
    # Team 2 info
    team2_id = row['team2_id']
    team2_name = team_name_map.get(team2_id, f"Unknown Team {team2_id}")
    team2_score = row['team2_score']
    team2_rank = row['team2_rank']
    
    # Create row for Team 1 perspective
    team_history.append({
        'dual_id': dual_id,
        'event_date': event_date,
        'team_id': team1_id,
        'team_name': team1_name,
        'team_rank': team1_rank,
        'team_score': team1_score,
        'opponent_id': team2_id,
        'opponent_name': team2_name,
        'opponent_rank': team2_rank,
        'opponent_score': team2_score,
        'result': 'Win' if team1_score > team2_score else 'Loss' if team1_score < team2_score else 'Tie',
        'score_diff': team1_score - team2_score,
        'year': event_date.year,
        'month': event_date.month,
        'day': event_date.day,
        'formatted_date': event_date.strftime('%Y-%m-%d')
    })
    
    # Create row for Team 2 perspective
    team_history.append({
        'dual_id': dual_id,
        'event_date': event_date,
        'team_id': team2_id,
        'team_name': team2_name,
        'team_rank': team2_rank,
        'team_score': team2_score,
        'opponent_id': team1_id,
        'opponent_name': team1_name,
        'opponent_rank': team1_rank,
        'opponent_score': team1_score,
        'result': 'Win' if team2_score > team1_score else 'Loss' if team2_score < team1_score else 'Tie',
        'score_diff': team2_score - team1_score,
        'year': event_date.year,
        'month': event_date.month,
        'day': event_date.day,
        'formatted_date': event_date.strftime('%Y-%m-%d')
    })

# Create DataFrame
team_history_df = pd.DataFrame(team_history)

# Sort by date for chronological order
team_history_df = team_history_df.sort_values(['event_date', 'dual_id'])

# Display sample
print("\n📊 Team Dual Meet History DataFrame Sample:")
print("="*80)
print(f"Total rows: {len(team_history_df)}")
print(f"Total unique duals: {len(team_history_df['dual_id'].unique())}")
print(f"Date range: {team_history_df['event_date'].min()} to {team_history_df['event_date'].max()}")
print(f"Teams covered: {team_history_df['team_name'].nunique()}")

print("\n📋 First 10 rows:")
display(team_history_df.head(10))

# Show available columns
print("\n📋 Columns available:")
print(team_history_df.columns.tolist())



📊 Team Dual Meet History DataFrame Sample:
Total rows: 1168
Total unique duals: 584
Date range: 2025-11-01 00:00:00 to 2026-02-23 00:00:00
Teams covered: 78

📋 First 10 rows:


,dual_id,event_date,team_id,team_name,team_rank,team_score,opponent_id,opponent_name,opponent_rank,opponent_score,result,score_diff,year,month,day,formatted_date
0,280,2025-11-01,24,Duke,72,19,67,Sacred Heart,76,18,Win,1,2025,11,1,2025-11-01
1,280,2025-11-01,67,Sacred Heart,76,18,24,Duke,72,19,Loss,-1,2025,11,1,2025-11-01
8,281,2025-11-01,27,Pittsburgh,19,20,68,Navy,21,12,Win,8,2025,11,1,2025-11-01
9,281,2025-11-01,68,Navy,21,12,27,Pittsburgh,19,20,Loss,-8,2025,11,1,2025-11-01
6,282,2025-11-01,18,Wisconsin,31,36,62,North Dakota State,33,3,Win,33,2025,11,1,2025-11-01
7,282,2025-11-01,62,North Dakota State,33,3,18,Wisconsin,31,36,Loss,-33,2025,11,1,2025-11-01
4,283,2025-11-01,70,Utah Valley,36,19,23,Stanford,11,12,Win,7,2025,11,1,2025-11-01
5,283,2025-11-01,23,Stanford,11,12,70,Utah Valley,36,19,Loss,-7,2025,11,1,2025-11-01
2,284,2025-11-01,20,Purdue,32,19,64,Cal Poly,47,13,Win,6,2025,11,1,2025-11-01
3,284,2025-11-01,64,Cal Poly,47,13,20,Purdue,32,19,Loss,-6,2025,11,1,2025-11-01



📋 Columns available:
['dual_id', 'event_date', 'team_id', 'team_name', 'team_rank', 'team_score', 'opponent_id', 'opponent_name', 'opponent_rank', 'opponent_score', 'result', 'score_diff', 'year', 'month', 'day', 'formatted_date']


In [19]:
team_history_df.to_csv('team_dual_history.csv', index=False)
print("\n💾 Saved to: team_dual_history.csv")
print("   Use this file in Tableau for team dual meet history visualization")


💾 Saved to: team_dual_history.csv
   Use this file in Tableau for team dual meet history visualization


In [20]:
team_history_df.tail()

,dual_id,event_date,team_id,team_name,team_rank,team_score,opponent_id,opponent_name,opponent_rank,opponent_score,result,score_diff,year,month,day,formatted_date
1143,560,2026-02-22,24,Duke,72,4,16,Cornell,13,42,Loss,-38,2026,2,22,2026-02-22
1140,561,2026-02-22,46,Bucknell,36,34,8,Bloomsburg,75,9,Win,25,2026,2,22,2026-02-22
1141,561,2026-02-22,8,Bloomsburg,75,9,46,Bucknell,36,34,Loss,-25,2026,2,22,2026-02-22
1166,547,2026-02-23,66,VMI,71,32,37,Presbyterian,78,13,Win,19,2026,2,23,2026-02-23
1167,547,2026-02-23,37,Presbyterian,78,13,66,VMI,71,32,Loss,-19,2026,2,23,2026-02-23


# Now we will look at individual wrestlers performances over the season

In [36]:
# ============================================
# NCAA QUALIFIER WRESTLER MATCHES - COMPLETE PROCESSING
# ============================================

import pandas as pd
import numpy as np
import re

# Read the matches data
matches = pd.read_csv('matches_updated (1).csv', parse_dates=['event_date']).sort_values(by='event_date')
teams = pd.read_csv('teams.xls')
wrestlers = pd.read_csv('wrestlers_updated.csv')

print("\n" + "="*100)
print("📊 NCAA QUALIFIER MATCHES PROCESSING")
print("="*100)

# ============================================
# STEP 1: HANDLE DUPLICATE WRESTLER NAMES
# ============================================

print("\n🔍 Handling duplicate wrestler names...")

# Create clean name fields
matches['home_name_clean'] = matches['home_name']
matches['away_name_clean'] = matches['away_name']

# Handle James Conway
matches.loc[(matches['home_name'] == 'James Conway') & (matches['home_team_name'] == 'Franklin & Marshall'), 'home_name_clean'] = 'James Conway (F&M)'
matches.loc[(matches['home_name'] == 'James Conway') & (matches['home_team_name'] == 'Missouri'), 'home_name_clean'] = 'James Conway (Mizzou)'
matches.loc[(matches['away_name'] == 'James Conway') & (matches['away_team_name'] == 'Franklin & Marshall'), 'away_name_clean'] = 'James Conway (F&M)'
matches.loc[(matches['away_name'] == 'James Conway') & (matches['away_team_name'] == 'Missouri'), 'away_name_clean'] = 'James Conway (Mizzou)'

# Handle Nathan Taylor
matches.loc[(matches['home_name'] == 'Nathan Taylor') & (matches['home_team_name'] == 'Lehigh'), 'home_name_clean'] = 'Nathan Taylor (Lehigh)'
matches.loc[(matches['home_name'] == 'Nathan Taylor') & (matches['home_team_name'] == 'Pennsylvania'), 'home_name_clean'] = 'Nathan Taylor (Penn)'
matches.loc[(matches['away_name'] == 'Nathan Taylor') & (matches['away_team_name'] == 'Lehigh'), 'away_name_clean'] = 'Nathan Taylor (Lehigh)'
matches.loc[(matches['away_name'] == 'Nathan Taylor') & (matches['away_team_name'] == 'Pennsylvania'), 'away_name_clean'] = 'Nathan Taylor (Penn)'

print("✅ Duplicate names handled")

# ============================================
# STEP 2: DEFINE NCAA QUALIFIERS FROM BRACKETS
# ============================================

print("\n📋 Defining NCAA qualifiers by weight class...")

# NCAA qualifiers from brackets (33 per weight class)
qualifiers_by_weight = {
    125: [
        'Jace Schafer', 'Mack Mauger', 'Luke Lilledahl', 'Jett Strickenberger', 
        'Ezekiel Witt', 'Maximo Renteria', 'Ayden Smith', 'Kael Lauridsen',
        'Dean Peterson', 'Troy Spratley', 'Andrew Binni', 'Conrad Hendriksen',
        'Vincent Robinson', 'Stevo Poulin', 'Diego Sotelo', 'Tyler Chappell',
        'Sheldon Seymour', 'Nic Bouzakis', 'Sulayman Bah', 'Kysen Terukina',
        'Jacob Moran', 'Tyler Klinsky', 'Davis Motyka', 'Brady Roark',
        'Jore Volk', 'Nico Provo', 'Cooper Flynn', 'Nicolar Rivera',
        'Marc-Anthony McGowan', 'Koda Holeman', 'Spencer Moore', 'Desmond Pleasant',
        'Eddie Ventresca'
    ],
    133: [
        'Carter Schmidt', 'Andrew Austin', 'Jax Forrest', 'T.K. Davis',
        'Zan Fugitt', 'Dominick Serrano', 'Blake Boarman', 'Will Betancourt',
        'Markel Baker', 'Kyler Larkin', 'Garrett Grice', 'Sean Spidle',
        'Evan Mougalian', 'Jacob Van Dee', 'Julian Farber', 'Luke Willochell',
        'Aaron Seidel', 'Marcus Blaze', 'Gabe Whisenhunt', 'Gage Walker',
        'Ethan Berginc', 'Tyler Ferrara', 'Zach Redding', 'Marcel Lopez',
        'Drake Ayala', 'Lucas Byrd', 'Dylan Shawver', 'Braxton Brown',
        'Maximilian Leete', 'Tyler Knox', 'Gunner Andrick', 'Gable Strickland',
        'Ben Davino'
    ],
    141: [
        'Aldo Hernandez', 'Matthew Martino', 'Jesse Mendez', 'Caedyn Ricciardi',
        'Ryan Jack', 'Joey Olivieri', 'Nash Singleton', 'Tom Crook',
        'Vance Vombaur', 'Luke Stanich', 'Pierson Manville', 'Tyler Wells',
        'Luke Simcox', 'Wyatt Henson', 'Julian Tagg', 'Jordan Titus',
        'Anthony Echemendia', 'Brock Hardy', 'Dario Lemus', 'Haiden Drury',
        'Braeden Davis', 'CJ Composto', 'Lorenzo Frezza', 'Gable Porter',
        'Vince Cornella', 'Nasir Bailey', 'Braden Basile', 'Dylan Chappell',
        'Jack Consiglio', 'Elijah Griffin', 'Carter Nogle', 'Billy DeKraker',
        'Sergio Vega'
    ],
    149: [
        'Austin McBurney', 'Clayton Jones', 'Shayne Van Ness', 'Lucas Kapusta',
        'Jacob Frost', 'David Evans', 'Andrew Clark', 'Michael Gioffre',
        'Casey Swiderski', 'Koy Buesgens', 'Kade Brown', 'Gabe Willochell',
        'Carter Young', 'Joseph Zargo', 'Chance Lamer', 'Kaden Cassidy',
        'Collin Gaj', 'Cross Wasilewski', 'Dylan Layton', 'Brock Herman',
        'Caleb Rathjen', 'Lachlan McNeil', 'Eligh Rivera', 'Andre Gonzales',
        'Caleb Tyus', 'Ethan Stiles', 'Anderson Heap', 'Max Petersen',
        'Aden Valencia', 'Ryder Block', 'Eugene Harney', 'Ryan Michaels',
        'Jaxon Joy'
    ],
    157: [
        'Yannis Charles', 'Jeb Prechtel', 'PJ Duke', 'Luke Mechler',
        'Cael Swensen', 'Daniel Cardenas', 'Jaivon Jones', 'Mason Shrader',
        'Brandon Cannon', 'Landon Robideau', 'Gavin Drexler', 'Charlie Millard',
        'Vinny Zerban', 'Derek Raike', 'Jimmy Harrington', 'Bryce Lowery',
        'Kaleb Larkin', 'Meyer Shapiro', 'Laird Root', 'Kai Owen',
        'Ethen Miller', 'Ty Watters', 'Colton Washleski', 'Dylan Evans',
        'Jude Swisher', 'Kannon Webster', 'Jonathan Ley', 'Kaleb Burgess',
        'Logan Rozynski', 'Cameron Catrabone', 'DJ McGee', 'Garrett McChesney',
        'Antrell Taylor'
    ],
    165: [
        'Ryan Vigil', 'Cody Walsh', 'Mitchell Mesenbrink', 'Braeden Scoles',
        'Paddy Gallagher', 'Bryce Hepner', 'Sean Seefeldt', 'Mac Church',
        'Matty Bianchi', 'LaDarion Lockett', 'Cody Goebel', 'Brock Woodcock',
        'Cesar Alvan', 'Andrew Sparks', 'Ty Whalen', 'Ryan Burgos',
        'Nicco Ruiz', 'Mikey Caliendo', 'Thomas Snipes', 'Noah Mulvaney',
        'Andrew Barbosa', 'Ryder Downey', 'Matthew Olguin', 'EJ Parco',
        'LJ Araujo', 'Max Brignola', 'Tyler Lillard', 'Chris Earnest',
        'Will Denny', 'Connor Euton', 'Gunner Filipowicz', 'Jared Keslar',
        'Joey Blaze'
    ],
    174: [
        "Grant O'Dell", 'Luke Condon', 'Levi Haines', 'Jared Simma',
        'Nick Fine', 'Beau Mantanona', 'Garrett Thompson', 'Sergio Desiante',
        'Alex Facundo', 'Patrick Kennedy', 'Holden Garcia', 'Lenny Pinto',
        'Carter Schubert', 'Carter Baer', 'Daschle Lamer', 'Avery Bassett',
        'Carson Kharchla', 'Christopher Minto', 'Riley Davis', 'Logan Messer',
        'Moses Espinoza-Owens', 'MJ Gaitan', 'Brody Baumann', 'Collin Carrigan',
        'Matty Singleton', 'Cam Steed', 'Derek Gilcher', 'Luca Augustine',
        'Myles Takats', 'Danny Wask', 'Colin Kelly', 'Cael Valencia',
        'Simon Ruiz'
    ],
    184: [
        'Sam Goin', 'Caleb Uhlenhopp', 'Rocco Welsh', 'Ian Bush',
        'Rylan Rogers', 'Chris Moore', 'Joe Curtis', 'Malachi DuVall',
        'Silas Allred', 'Brock Mantanona', 'Abraham Wojcikiewicz', 'Tomas Brooker',
        'Dylan Fishback', 'Isaac Dean', 'Brian Soldano', 'Nick Fox',
        'James Conway (F&M)', 'Max McEnelly', 'Tyler Bienus', 'Jared McGill',
        'Jaden Bullock', 'Shane Cartagena-Walsh', 'Zack Ryder', 'Aidan Brenot',
        'Eddie Neitenbach', 'Angelo Ferrari', 'Chase Kranitz', 'Ceasar Garza',
        'Caleb Campos', 'Sal Perrine', 'Jake Dailey', 'Mahonri Rushton',
        'Aeoden Sinclair'
    ],
    197: [
        'Karson Tompkins', 'Blake Schaffer', 'Josh Barr', 'Dillon Bechtold',
        'Branson John', 'Angelo Posada', 'Brock Zurawski', 'Evan Bates',
        'DJ Parker', 'Joey Novak', 'Kael Wisler', 'Rune Lawrence',
        'Luke Geog', 'Bennett Berge', 'Wyatt Ingham', 'Colton Hawks',
        'Sonny Sasso', 'Stephen Little', 'Kade Rule', 'Zayne Lehman',
        'Gabe Sollars', 'Camden McDanel', 'Devin Wasley', 'Gabe Arnold',
        'Justin Rademacher', 'Cody Merrill', 'Ben Vanadia', 'Mikey Squires',
        'Mac Stout', 'Remy Cotton', 'Andrew Reall', 'Kael Bennie',
        'Rocky Elam'
    ],
    285: [
        'Mason Rebuck', 'Emmanuel Ulrich', 'Yonger Bastida', 'Vincent Mueller',
        'Jimmy Mullen', 'Cole Mirasola', 'Connor Barket', 'Alex Semenenko',
        'Ben Kueter', 'Nick Feldman', 'Jarrett Stoner', 'Juan Mora',
        'Braxton Amos', 'Spencer Lanosga', 'Dayton Pitzer', 'Luke Rasmussen',
        'AJ Ferrari', 'Taye Ghadiali', 'Jack Forbes', 'Nate Schon',
        'Koy Hopke', 'Devon Dawson', 'Trevor Tinker', 'Hunter Catka',
        'Nathan Taylor (Lehigh)', 'Konner Doucet', 'Luke Luffman', 'Stephan Monchery',
        'David Szuba', 'Brady Colbert', 'Christian Carroll', 'Brenan Morgan',
        'Isaac Trumble'
    ]
}

# Create qualifier set for quick lookup
qualifier_set = set()
for weight, names in qualifiers_by_weight.items():
    for name in names:
        qualifier_set.add(name)

print(f"✅ Total unique qualifiers: {len(qualifier_set)}")

# ============================================
# STEP 3: IDENTIFY SPECIAL MATCHES (Injury, DQ, Forfeit)
# ============================================

print("\n🔍 Identifying special match outcomes...")

def categorize_win_type(win_type):
    """Categorize win type into broader categories"""
    if pd.isna(win_type):
        return 'Unknown'
    
    win_type_str = str(win_type).upper()
    
    if 'INJ' in win_type_str:
        return 'Injury Default'
    elif 'DQ' in win_type_str:
        return 'Disqualification'
    elif 'MFF' in win_type_str or 'FOR' in win_type_str:
        return 'Forfeit'
    elif 'FALL' in win_type_str:
        return 'Fall'
    elif 'TF' in win_type_str:
        return 'Tech Fall'
    elif 'MD' in win_type_str:
        return 'Major Decision'
    elif 'DEC' in win_type_str or 'SV' in win_type_str or 'TB' in win_type_str:
        return 'Decision'
    else:
        return 'Other'

matches['win_category'] = matches['win_type'].apply(categorize_win_type)
matches['is_special'] = matches['win_category'].isin(['Injury Default', 'Disqualification', 'Forfeit'])

print(f"✅ Special matches identified: {matches['is_special'].sum()}")
print(f"   - Forfeits: {(matches['win_category'] == 'Forfeit').sum()}")
print(f"   - Injury Defaults: {(matches['win_category'] == 'Injury Default').sum()}")
print(f"   - Disqualifications: {(matches['win_category'] == 'Disqualification').sum()}")

# ============================================
# STEP 4: EXTRACT SCORES FROM RESULT STRINGS
# ============================================

print("\n🔍 Extracting scores from match results...")

def extract_scores(row):
    """Extract home and away scores from result string"""
    result_str = str(row['Result'])
    home_win = row['home_win']
    win_category = row['win_category']
    
    # Default values
    home_score = None
    away_score = None
    
    # For special matches, we might not have scores
    if win_category in ['Injury Default', 'Disqualification', 'Forfeit']:
        return home_score, away_score
    
    # Skip falls and tech falls (they don't have standard scores)
    if 'FALL' not in result_str.upper() and 'TF' not in result_str.upper():
        try:
            # Find all numbers in the result string
            numbers = re.findall(r'\d+', result_str)
            if len(numbers) >= 2:
                score1 = int(numbers[0])
                score2 = int(numbers[1])
                
                # Assign based on winner
                if home_win:
                    home_score = score1
                    away_score = score2
                else:
                    home_score = score2
                    away_score = score1
        except:
            pass
    
    return home_score, away_score

# Apply score extraction
scores = matches.apply(extract_scores, axis=1, result_type='expand')
matches['home_score'] = scores[0]
matches['away_score'] = scores[1]

print("✅ Score extraction complete")

# ============================================
# STEP 5: CREATE TABLEAU-FRIENDLY LONG FORMAT
# ============================================

print("\n📊 Creating long format for Tableau...")

def create_wrestler_record(perspective, row, match_id, dual_id, weight_class, event_date, formatted_date, year, month,
                          wrestler_name, wrestler_team, wrestler_id, wrestler_class, is_qualifier,
                          opponent_name, opponent_team, opponent_id, opponent_class, opponent_is_qualifier,
                          opponent_display, wrestler_score, opponent_score, score_display, result,
                          win_type, win_category, result_detail, is_special):
    
    return {
        # Match identifiers
        'match_id': match_id,
        'dual_id': dual_id,
        'weight_class': weight_class,
        'event_date': event_date,
        'formatted_date': formatted_date,
        'year': year,
        'month': month,
        
        # Wrestler info
        'wrestler_id': wrestler_id,
        'wrestler_name': wrestler_name,
        'wrestler_team': wrestler_team,
        'wrestler_class': wrestler_class,
        'is_qualifier': is_qualifier,
        
        # Opponent info
        'opponent_id': opponent_id,
        'opponent_display': opponent_display,
        'opponent_name': opponent_name,
        'opponent_team': opponent_team,
        'opponent_class': opponent_class,
        'opponent_is_qualifier': opponent_is_qualifier,
        
        # Match outcome
        'wrestler_score': wrestler_score,
        'opponent_score': opponent_score,
        'score_display': score_display,
        'result': 'Win' if result else 'Loss' if result is not None else 'Unknown',
        'win_type': win_type,
        'win_category': win_category,
        'result_detail': result_detail,
        'is_special': is_special,
        
        # Match flags
        'is_qualifier_match': is_qualifier and opponent_is_qualifier,
        'match_type': 'Qualifier vs Qualifier' if is_qualifier and opponent_is_qualifier else 'Qualifier vs Non-Qualifier'
    }

ncaa_records = []

for idx, row in matches.iterrows():
    dual_id = row['dual_id']
    weight_class = row['weight_class']
    event_date = row['event_date']
    win_type = row['win_type']
    win_category = row['win_category']
    is_forfeit = win_category == 'Forfeit'
    
    # Format date nicely
    if pd.notna(event_date):
        try:
            formatted_date = pd.to_datetime(event_date).strftime('%Y-%m-%d')
            year = pd.to_datetime(event_date).year
            month = pd.to_datetime(event_date).month
        except:
            formatted_date = str(event_date)
            year = None
            month = None
    else:
        formatted_date = None
        year = None
        month = None
    
    # Home wrestler info
    home_name = row['home_name_clean'] if pd.notna(row['home_name_clean']) else "Unknown"
    home_team = row['home_team_name'] if pd.notna(row['home_team_name']) else "Unknown Team"
    home_id = row['home_wrestler_id'] if pd.notna(row['home_wrestler_id']) else None
    home_class = row['home_class'] if pd.notna(row['home_class']) else None
    home_is_qualifier = home_name in qualifier_set if home_name != "Unknown" else False
    
    # Away wrestler info
    away_name = row['away_name_clean'] if pd.notna(row['away_name_clean']) else "Unknown"
    away_team = row['away_team_name'] if pd.notna(row['away_team_name']) else "Unknown Team"
    away_id = row['away_wrestler_id'] if pd.notna(row['away_wrestler_id']) else None
    away_class = row['away_class'] if pd.notna(row['away_class']) else None
    away_is_qualifier = away_name in qualifier_set if away_name != "Unknown" else False
    
    # For forfeits, handle the missing wrestler specially
    if is_forfeit:
        if row['home_win']:
            # Home won, so away forfeited
            away_name = f"FORFEIT"
            away_team = "N/A"
            away_is_qualifier = False
        else:
            # Away won, so home forfeited
            home_name = f"FORFEIT"
            home_team = "N/A"
            home_is_qualifier = False
    
    # Create opponent display strings
    if away_is_qualifier:
        home_opponent_display = f"{away_name} ({away_team})"
    else:
        if away_name == "FORFEIT":
            home_opponent_display = "FORFEIT"
        elif away_name == "Unknown":
            home_opponent_display = "Unknown Opponent"
        else:
            home_opponent_display = f"Non-Qualifier: {away_name}" + (f" ({away_team})" if away_team != "Unknown Team" else "")
    
    if home_is_qualifier:
        away_opponent_display = f"{home_name} ({home_team})"
    else:
        if home_name == "FORFEIT":
            away_opponent_display = "FORFEIT"
        elif home_name == "Unknown":
            away_opponent_display = "Unknown Opponent"
        else:
            away_opponent_display = f"Non-Qualifier: {home_name}" + (f" ({home_team})" if home_team != "Unknown Team" else "")
    
    # Match result info
    home_win = row['home_win']
    result_detail = row['Result']
    is_special = row['is_special']
    
    # Scores
    home_score = row['home_score'] if pd.notna(row['home_score']) else None
    away_score = row['away_score'] if pd.notna(row['away_score']) else None
    
    # Create display score with special match indicator
    if is_special:
        if win_category == 'Injury Default':
            score_display = f"INJURY DEFAULT - {result_detail}"
        elif win_category == 'Disqualification':
            score_display = f"DQ - {result_detail}"
        elif win_category == 'Forfeit':
            score_display = f"FORFEIT - {result_detail}"
        else:
            score_display = result_detail
    else:
        if home_score is not None and away_score is not None:
            score_display = f"{home_score}-{away_score}"
        else:
            score_display = win_type if pd.notna(win_type) else "No Score"
    
    # Create match ID
    match_id = f"{dual_id}_{weight_class}"
    
    # Create record for home wrestler (if they actually wrestled)
    if home_name != "Unknown" and home_name != "FORFEIT":
        ncaa_records.append(create_wrestler_record(
            'home', row, match_id, dual_id, weight_class, event_date, formatted_date, year, month,
            home_name, home_team, home_id, home_class, home_is_qualifier,
            away_name, away_team, away_id, away_class, away_is_qualifier,
            home_opponent_display, home_score, away_score, score_display, home_win,
            win_type, win_category, result_detail, is_special
        ))
    
    # Create record for away wrestler (if they actually wrestled)
    if away_name != "Unknown" and away_name != "FORFEIT":
        ncaa_records.append(create_wrestler_record(
            'away', row, match_id, dual_id, weight_class, event_date, formatted_date, year, month,
            away_name, away_team, away_id, away_class, away_is_qualifier,
            home_name, home_team, home_id, home_class, home_is_qualifier,
            away_opponent_display, away_score, home_score, score_display, not home_win if home_win is not None else None,
            win_type, win_category, result_detail, is_special
        ))

# Create DataFrame
ncaa_df = pd.DataFrame(ncaa_records)

# Sort by date
ncaa_df = ncaa_df.sort_values(['event_date', 'match_id']).reset_index(drop=True)

print(f"\n✅ Created {len(ncaa_df)} rows")
print(f"   {ncaa_df['match_id'].nunique()} unique matches")
print(f"   {ncaa_df['wrestler_name'].nunique()} unique wrestlers")

# ============================================
# STEP 6: FIX ANY REMAINING NULLS FOR TABLEAU
# ============================================

print("\n🔧 Fixing remaining nulls for Tableau filtering...")

# Fill any remaining nulls in key fields
ncaa_df['wrestler_name'] = ncaa_df['wrestler_name'].fillna('Unknown Wrestler')
ncaa_df['wrestler_team'] = ncaa_df['wrestler_team'].fillna('Unknown Team')
ncaa_df['wrestler_class'] = ncaa_df['wrestler_class'].fillna('Unknown')

# Create filter field for wrestler dropdown (guaranteed no nulls)
ncaa_df['wrestler_filter'] = ncaa_df['wrestler_name'] + ' (' + ncaa_df['weight_class'].astype(str) + ' lbs)'

# Create opponent filter field (guaranteed no nulls)
ncaa_df['opponent_filter'] = ncaa_df['opponent_display'].fillna('Unknown Opponent')

# Create match description for tooltips
ncaa_df['match_description'] = ncaa_df.apply(
    lambda x: f"{x['wrestler_name']} vs {x['opponent_display']} - {x['score_display']}", 
    axis=1
)

print("✅ Nulls fixed")
print(f"   Nulls in wrestler_name: {ncaa_df['wrestler_name'].isna().sum()}")
print(f"   Nulls in wrestler_filter: {ncaa_df['wrestler_filter'].isna().sum()}")

# ============================================
# STEP 7: CREATE WRESTLER LOOKUP TABLE
# ============================================

print("\n📋 Creating wrestler lookup table...")

# Get all unique wrestlers
wrestler_lookup = ncaa_df[['wrestler_name', 'wrestler_team', 'weight_class', 'is_qualifier']].drop_duplicates()
wrestler_lookup = wrestler_lookup[wrestler_lookup['wrestler_name'] != 'Unknown Wrestler']
wrestler_lookup = wrestler_lookup.sort_values(['weight_class', 'wrestler_name']).reset_index(drop=True)

# Create display names
wrestler_lookup['display_name'] = wrestler_lookup.apply(
    lambda x: f"{x['wrestler_name']} ({x['weight_class']} lbs)" + 
              (f" - Qualifier" if x['is_qualifier'] else f" - Non-Qualifier"), 
    axis=1
)

print(f"✅ Created lookup for {len(wrestler_lookup)} wrestlers")
print(f"   Qualifiers: {len(wrestler_lookup[wrestler_lookup['is_qualifier']])}")
print(f"   Non-qualifiers: {len(wrestler_lookup[~wrestler_lookup['is_qualifier']])}")

# ============================================
# STEP 8: CREATE SUMMARY STATISTICS
# ============================================

print("\n📊 Creating summary statistics...")

# Matches by weight class
weight_summary = ncaa_df.groupby('weight_class').agg({
    'match_id': 'nunique',
    'wrestler_name': 'nunique',
    'is_qualifier_match': 'sum'
}).rename(columns={
    'match_id': 'total_matches',
    'wrestler_name': 'unique_wrestlers',
    'is_qualifier_match': 'qualifier_match_count'
}).reset_index()

# Add special match counts
special_counts = ncaa_df[ncaa_df['is_special']].groupby('weight_class').size().reset_index(name='special_matches')
weight_summary = weight_summary.merge(special_counts, on='weight_class', how='left').fillna(0)
weight_summary['special_matches'] = weight_summary['special_matches'].astype(int)

print("✅ Summary created")

# ============================================
# STEP 9: EXPORT FOR TABLEAU
# ============================================

# print("\n💾 Exporting files for Tableau...")

# # Main match data
# ncaa_df.to_csv('ncaa_wrestler_matches.csv', index=False)
# print("   ✅ Saved: ncaa_wrestler_matches.csv")

# # Wrestler lookup for filters
# wrestler_lookup.to_csv('ncaa_wrestler_lookup.csv', index=False)
# print("   ✅ Saved: ncaa_wrestler_lookup.csv")

# # Weight class summary
# weight_summary.to_csv('ncaa_weight_summary.csv', index=False)
# print("   ✅ Saved: ncaa_weight_summary.csv")

# print("\n" + "="*100)
# print("✅ PROCESSING COMPLETE")
# print("="*100)
# print("\n📌 KEY FEATURES:")
# print("   - Forfeits handled: losing wrestler shows as 'FORFEIT'")
# print("   - No nulls in wrestler_filter field (use this for dropdowns)")
# print("   - Opponent display includes team and qualifier status")
# print("   - Special matches clearly labeled (INJURY/DQ/FORFEIT)")
# print("   - Ranks removed from output")

# print("\n📌 TABLEAU SETUP:")
# print("   1. Connect to ncaa_wrestler_matches.csv")
# print("   2. For Weight Class filter: Drag 'weight_class' to Filters → Show Filter")
# print("   3. For Wrestler filter: Drag 'wrestler_filter' to Filters → Show Filter")
# print("   4. Right-click wrestler filter → 'Only Relevant Values'")
# print("   5. Build table with:")
# print("      - Rows: formatted_date")
# print("      - Rows: opponent_filter")
# print("      - Text: score_display")
# print("      - Color: result (Win=Green, Loss=Red)")
# print("      - Tooltip: win_category, is_special, match_type")


📊 NCAA QUALIFIER MATCHES PROCESSING

🔍 Handling duplicate wrestler names...
✅ Duplicate names handled

📋 Defining NCAA qualifiers by weight class...
✅ Total unique qualifiers: 330

🔍 Identifying special match outcomes...
✅ Special matches identified: 208
   - Forfeits: 146
   - Injury Defaults: 53
   - Disqualifications: 9

🔍 Extracting scores from match results...
✅ Score extraction complete

📊 Creating long format for Tableau...

✅ Created 11534 rows
   5840 unique matches
   1512 unique wrestlers

🔧 Fixing remaining nulls for Tableau filtering...
✅ Nulls fixed
   Nulls in wrestler_name: 0
   Nulls in wrestler_filter: 0

📋 Creating wrestler lookup table...
✅ Created lookup for 1704 wrestlers
   Qualifiers: 326
   Non-qualifiers: 1378

📊 Creating summary statistics...
✅ Summary created


In [38]:
ncaa_df

,match_id,dual_id,weight_class,event_date,formatted_date,year,month,wrestler_id,wrestler_name,wrestler_team,wrestler_class,is_qualifier,opponent_id,opponent_display,opponent_name,opponent_team,opponent_class,opponent_is_qualifier,wrestler_score,opponent_score,score_display,result,win_type,win_category,result_detail,is_special,is_qualifier_match,match_type,wrestler_filter,opponent_filter,match_description
0,280_125,280,125,2025-11-01,2025-11-01,2025,11,266.0,Riley Rowan,Duke,RSFR,False,1254.0,Non-Qualifier: Joey Clem (Sacred Heart),Joey Clem,Sacred Heart,RSFR,False,2.0,0.0,2.0-0.0,Win,WDEC,Decision,WDEC 2 - 0,False,False,Qualifier vs Non-Qualifier,Riley Rowan (125 lbs),Non-Qualifier: Joey Clem (Sacred Heart),Riley Rowan vs Non-Qualifier: Joey Clem (Sacre...
1,280_125,280,125,2025-11-01,2025-11-01,2025,11,1254.0,Joey Clem,Sacred Heart,RSFR,False,266.0,Non-Qualifier: Riley Rowan (Duke),Riley Rowan,Duke,RSFR,False,0.0,2.0,2.0-0.0,Loss,WDEC,Decision,WDEC 2 - 0,False,False,Qualifier vs Non-Qualifier,Joey Clem (125 lbs),Non-Qualifier: Riley Rowan (Duke),Joey Clem vs Non-Qualifier: Riley Rowan (Duke)...
2,280_133,280,133,2025-11-01,2025-11-01,2025,11,1080.0,John King,Duke,RSFR,False,1212.0,Non-Qualifier: Braxton Appello (Sacred Heart),Braxton Appello,Sacred Heart,JR,False,NaN,NaN,LTF,Loss,LTF,Tech Fall,LTF5 18 - 2 3:00,False,False,Qualifier vs Non-Qualifier,John King (133 lbs),Non-Qualifier: Braxton Appello (Sacred Heart),John King vs Non-Qualifier: Braxton Appello (S...
3,280_133,280,133,2025-11-01,2025-11-01,2025,11,1212.0,Braxton Appello,Sacred Heart,JR,False,1080.0,Non-Qualifier: John King (Duke),John King,Duke,RSFR,False,NaN,NaN,LTF,Win,LTF,Tech Fall,LTF5 18 - 2 3:00,False,False,Qualifier vs Non-Qualifier,Braxton Appello (133 lbs),Non-Qualifier: John King (Duke),Braxton Appello vs Non-Qualifier: John King (D...
4,280_141,280,141,2025-11-01,2025-11-01,2025,11,267.0,Raymond Adams,Duke,JR,False,825.0,Non-Qualifier: John Hildebrandt (Sacred Heart),John Hildebrandt,Sacred Heart,JR,False,2.0,3.0,2.0-3.0,Loss,LDEC,Decision,LDEC 3 - 2,False,False,Qualifier vs Non-Qualifier,Raymond Adams (141 lbs),Non-Qualifier: John Hildebrandt (Sacred Heart),Raymond Adams vs Non-Qualifier: John Hildebran...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11529,547_184,547,184,2026-02-23,2026-02-23,2026,2,815.0,Andrew Barford,VMI,FR,False,400.0,Non-Qualifier: Reed Douglass (Presbyterian),Reed Douglass,Presbyterian,JR,False,NaN,NaN,WTF,Loss,WTF,Tech Fall,WTF5 22 - 5 6:45,False,False,Qualifier vs Non-Qualifier,Andrew Barford (184 lbs),Non-Qualifier: Reed Douglass (Presbyterian),Andrew Barford vs Non-Qualifier: Reed Douglass...
11530,547_197,547,197,2026-02-23,2026-02-23,2026,2,911.0,Toler Hornick,Presbyterian,SO,False,816.0,Non-Qualifier: Toby Schoffstall (VMI),Toby Schoffstall,VMI,JR,False,NaN,NaN,LTF,Loss,LTF,Tech Fall,LTF5 15 - 0 1:05,False,False,Qualifier vs Non-Qualifier,Toler Hornick (197 lbs),Non-Qualifier: Toby Schoffstall (VMI),Toler Hornick vs Non-Qualifier: Toby Schoffsta...
11531,547_197,547,197,2026-02-23,2026-02-23,2026,2,816.0,Toby Schoffstall,VMI,JR,False,911.0,Non-Qualifier: Toler Hornick (Presbyterian),Toler Hornick,Presbyterian,SO,False,NaN,NaN,LTF,Win,LTF,Tech Fall,LTF5 15 - 0 1:05,False,False,Qualifier vs Non-Qualifier,Toby Schoffstall (197 lbs),Non-Qualifier: Toler Hornick (Presbyterian),Toby Schoffstall vs Non-Qualifier: Toler Horni...
11532,547_285,547,285,2026-02-23,2026-02-23,2026,2,1484.0,Ryland Whitworth,Presbyterian,FR,False,1468.0,Non-Qualifier: Drake Garrison (VMI),Drake Garrison,VMI,SO,False,NaN,NaN,WTF,Win,WTF,Tech Fall,WTF5 21 - 5 5:00,False,False,Qualifier vs Non-Qualifier,Ryland Whitworth (285 lbs),Non-Qualifier: Drake Garrison (VMI),Ryland Whitworth vs Non-Qualifier: Drake Garri...


In [39]:
ncaa_df.query('wrestler_name == "Drake Ayala"')

,match_id,dual_id,weight_class,event_date,formatted_date,year,month,wrestler_id,wrestler_name,wrestler_team,wrestler_class,is_qualifier,opponent_id,opponent_display,opponent_name,opponent_team,opponent_class,opponent_is_qualifier,wrestler_score,opponent_score,score_display,result,win_type,win_category,result_detail,is_special,is_qualifier_match,match_type,wrestler_filter,opponent_filter,match_description
182,275_133,275,133,2025-11-06,2025-11-06,2025,11,196.0,Drake Ayala,Iowa,SR,True,1005.0,Non-Qualifier: Trayce Eckman (Bellarmine),Trayce Eckman,Bellarmine,JR,False,NaN,NaN,WTF,Win,WTF,Tech Fall,WTF5 19 - 4 4:28,False,False,Qualifier vs Non-Qualifier,Drake Ayala (133 lbs),Non-Qualifier: Trayce Eckman (Bellarmine),Drake Ayala vs Non-Qualifier: Trayce Eckman (B...
797,206_133,206,133,2025-11-15,2025-11-15,2025,11,196.0,Drake Ayala,Iowa,SR,True,140.0,Lucas Byrd (Illinois),Lucas Byrd,Illinois,SR,True,2.0,7.0,2.0-7.0,Loss,LDEC,Decision,LDEC 7 - 2,False,True,Qualifier vs Qualifier,Drake Ayala (133 lbs),Lucas Byrd (Illinois),Drake Ayala vs Lucas Byrd (Illinois) - 2.0-7.0
1057,219_133,219,133,2025-11-15,2025-11-15,2025,11,196.0,Drake Ayala,Iowa,SR,True,928.0,Non-Qualifier: Kade Moore (Missouri),Kade Moore,Missouri,JR,False,NaN,NaN,WFALL,Win,WFALL,Fall,WFALL 3:57,False,False,Qualifier vs Non-Qualifier,Drake Ayala (133 lbs),Non-Qualifier: Kade Moore (Missouri),Drake Ayala vs Non-Qualifier: Kade Moore (Miss...
1573,187_133,187,133,2025-11-16,2025-11-16,2025,11,196.0,Drake Ayala,Iowa,SR,True,591.0,Non-Qualifier: Ronnie Ramirez (Oklahoma State),Ronnie Ramirez,Oklahoma State,FR,False,1.0,7.0,1.0-7.0,Win,WSV,Decision,WSV-1 7 - 4,False,False,Qualifier vs Non-Qualifier,Drake Ayala (133 lbs),Non-Qualifier: Ronnie Ramirez (Oklahoma State),Drake Ayala vs Non-Qualifier: Ronnie Ramirez (...
1593,188_133,188,133,2025-11-16,2025-11-16,2025,11,196.0,Drake Ayala,Iowa,SR,True,443.0,Ben Davino (Ohio State),Ben Davino,Ohio State,RSFR,True,4.0,10.0,10.0-4.0,Loss,WDEC,Decision,WDEC 10 - 4,False,True,Qualifier vs Qualifier,Drake Ayala (133 lbs),Ben Davino (Ohio State),Drake Ayala vs Ben Davino (Ohio State) - 10.0-4.0
2007,178_133,178,133,2025-11-21,2025-11-21,2025,11,196.0,Drake Ayala,Iowa,SR,True,295.0,Non-Qualifier: Evan Tallmadge (Pittsburgh),Evan Tallmadge,Pittsburgh,SO,False,13.0,4.0,13.0-4.0,Win,WMD,Major Decision,WMD 13 - 4,False,False,Qualifier vs Non-Qualifier,Drake Ayala (133 lbs),Non-Qualifier: Evan Tallmadge (Pittsburgh),Drake Ayala vs Non-Qualifier: Evan Tallmadge (...
2403,162_133,162,133,2025-11-30,2025-11-30,2025,11,196.0,Drake Ayala,Iowa,SR,True,600.0,Non-Qualifier: Evan Frost (Iowa State),Evan Frost,Iowa State,JR,False,5.0,11.0,11.0-5.0,Loss,WDEC,Decision,WDEC 11 - 5,False,False,Qualifier vs Non-Qualifier,Drake Ayala (133 lbs),Non-Qualifier: Evan Frost (Iowa State),Drake Ayala vs Non-Qualifier: Evan Frost (Iowa...
5143,29_133,29,133,2026-01-09,2026-01-09,2026,1,196.0,Drake Ayala,Iowa,SR,True,206.0,Zan Fugitt (Wisconsin),Zan Fugitt,Wisconsin,SO,True,5.0,6.0,5.0-6.0,Loss,LDEC,Decision,LDEC 6 - 5,False,True,Qualifier vs Qualifier,Drake Ayala (133 lbs),Zan Fugitt (Wisconsin),Drake Ayala vs Zan Fugitt (Wisconsin) - 5.0-6.0
5983,324_133,324,133,2026-01-16,2026-01-16,2026,1,196.0,Drake Ayala,Iowa,SR,True,120.0,Marcus Blaze (Penn State),Marcus Blaze,Penn State,FR,True,2.0,4.0,2.0-4.0,Loss,LDEC,Decision,LDEC 4 - 2,False,True,Qualifier vs Qualifier,Drake Ayala (133 lbs),Marcus Blaze (Penn State),Drake Ayala vs Marcus Blaze (Penn State) - 2.0...
6985,373_133,373,133,2026-01-23,2026-01-23,2026,1,196.0,Drake Ayala,Iowa,SR,True,216.0,Jacob Van Dee (Nebraska),Jacob Van Dee,Nebraska,JR,True,12.0,6.0,6.0-12.0,Win,LDEC,Decision,LDEC 12 - 6,False,True,Qualifier vs Qualifier,Drake Ayala (133 lbs),Jacob Van Dee (Nebraska),Drake Ayala vs Jacob Van Dee (Nebraska) - 6.0-...


In [40]:
ncaa_df.isna().sum()

match_id                    0
dual_id                     0
weight_class                0
event_date                  0
formatted_date              0
year                        0
month                       0
wrestler_id                 0
wrestler_name               0
wrestler_team               0
wrestler_class              0
is_qualifier                0
opponent_id               146
opponent_display            0
opponent_name               0
opponent_team               0
opponent_class            146
opponent_is_qualifier       0
wrestler_score           3364
opponent_score           3364
score_display               0
result                      0
win_type                    0
win_category                0
result_detail               0
is_special                  0
is_qualifier_match          0
match_type                  0
wrestler_filter             0
opponent_filter             0
match_description           0
dtype: int64

In [41]:
ncaa_df.to_csv('ncaa_wrestler_matches.csv', index=False)

In [50]:
ncaa_df.query('wrestler_name.str.contains("Arnold")', engine="python")

,match_id,dual_id,weight_class,event_date,formatted_date,year,month,wrestler_id,wrestler_name,wrestler_team,wrestler_class,is_qualifier,opponent_id,opponent_display,opponent_name,opponent_team,opponent_class,opponent_is_qualifier,wrestler_score,opponent_score,score_display,result,win_type,win_category,result_detail,is_special,is_qualifier_match,match_type,wrestler_filter,opponent_filter,match_description
2019,178_184,178,184,2025-11-21,2025-11-21,2025,11,992.0,Gabe Arnold,Iowa,SO,True,301.0,Chase Kranitz (Pittsburgh),Chase Kranitz,Pittsburgh,JR,True,15.0,5.0,15.0-5.0,Win,WMD,Major Decision,WMD 15 - 5,False,True,Qualifier vs Qualifier,Gabe Arnold (184 lbs),Chase Kranitz (Pittsburgh),Gabe Arnold vs Chase Kranitz (Pittsburgh) - 15...
3053,127_197,127,197,2025-12-12,2025-12-12,2025,12,992.0,Gabe Arnold,Iowa,SO,True,878.0,Kael Bennie (Utah Valley),Kael Bennie,Utah Valley,SO,True,4.0,2.0,4.0-2.0,Win,WDEC,Decision,WDEC 4 - 2,False,True,Qualifier vs Qualifier,Gabe Arnold (197 lbs),Kael Bennie (Utah Valley),Gabe Arnold vs Kael Bennie (Utah Valley) - 4.0...
3073,128_197,128,197,2025-12-12,2025-12-12,2025,12,992.0,Gabe Arnold,Iowa,SO,True,997.0,Kade Rule (Chattanooga),Kade Rule,Chattanooga,FR,True,17.0,4.0,17.0-4.0,Win,WMD,Major Decision,WMD 17 - 4,False,True,Qualifier vs Qualifier,Gabe Arnold (197 lbs),Kade Rule (Chattanooga),Gabe Arnold vs Kade Rule (Chattanooga) - 17.0-4.0
5993,324_174,324,174,2026-01-16,2026-01-16,2026,1,992.0,Gabe Arnold,Iowa,SO,True,125.0,Levi Haines (Penn State),Levi Haines,Penn State,SR,True,2.0,4.0,2.0-4.0,Loss,LDEC,Decision,LDEC 4 - 2,False,True,Qualifier vs Qualifier,Gabe Arnold (174 lbs),Levi Haines (Penn State),Gabe Arnold vs Levi Haines (Penn State) - 2.0-4.0
6997,373_184,373,184,2026-01-23,2026-01-23,2026,1,992.0,Gabe Arnold,Iowa,SO,True,222.0,Silas Allred (Nebraska),Silas Allred,Nebraska,SR,True,4.0,1.0,1.0-4.0,Win,LDEC,Decision,LDEC 4 - 1,False,True,Qualifier vs Qualifier,Gabe Arnold (184 lbs),Silas Allred (Nebraska),Gabe Arnold vs Silas Allred (Nebraska) - 1.0-4.0
7909,413_184,413,184,2026-01-30,2026-01-30,2026,1,992.0,Gabe Arnold,Iowa,SO,True,156.0,Max McEnelly (Minnesota),Max McEnelly,Minnesota,SO,True,1.0,4.0,1.0-4.0,Loss,LDEC,Decision,LDEC 4 - 1,False,True,Qualifier vs Qualifier,Gabe Arnold (184 lbs),Max McEnelly (Minnesota),Gabe Arnold vs Max McEnelly (Minnesota) - 1.0-4.0
8882,511_184,511,184,2026-02-06,2026-02-06,2026,2,992.0,Gabe Arnold,Iowa,SO,True,449.0,Dylan Fishback (Ohio State),Dylan Fishback,Ohio State,JR,True,4.0,1.0,1.0-4.0,Loss,WSV,Decision,WSV-1 4 - 1,False,True,Qualifier vs Qualifier,Gabe Arnold (184 lbs),Dylan Fishback (Ohio State),Gabe Arnold vs Dylan Fishback (Ohio State) - 1...
9653,500_184,500,184,2026-02-08,2026-02-08,2026,2,992.0,Gabe Arnold,Iowa,SO,True,291.0,Non-Qualifier: Ryan Boucher (Michigan State),Ryan Boucher,Michigan State,SR,False,NaN,NaN,LTF,Win,LTF,Tech Fall,LTF5 20 - 4 7:00,False,False,Qualifier vs Non-Qualifier,Gabe Arnold (184 lbs),Non-Qualifier: Ryan Boucher (Michigan State),Gabe Arnold vs Non-Qualifier: Ryan Boucher (Mi...
9905,470_184,470,184,2026-02-13,2026-02-13,2026,2,992.0,Gabe Arnold,Iowa,SO,True,281.0,Brock Mantanona (Michigan),Brock Mantanona,Michigan,RSFR,True,1.0,3.0,1.0-3.0,Win,WTB,Decision,WTB-1 3 - 2,False,True,Qualifier vs Qualifier,Gabe Arnold (184 lbs),Brock Mantanona (Michigan),Gabe Arnold vs Brock Mantanona (Michigan) - 1....
10502,444_184,444,184,2026-02-15,2026-02-15,2026,2,992.0,Gabe Arnold,Iowa,SO,True,232.0,Non-Qualifier: James Rowley (Purdue),James Rowley,Purdue,JR,False,4.0,1.0,1.0-4.0,Win,LDEC,Decision,LDEC 4 - 1,False,False,Qualifier vs Non-Qualifier,Gabe Arnold (184 lbs),Non-Qualifier: James Rowley (Purdue),Gabe Arnold vs Non-Qualifier: James Rowley (Pu...


In [49]:
matches.query('home_name == "Gabe Arnold" or away_name == "Gabe Arnold"')

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name,home_name_clean,away_name_clean,win_category,is_special,home_score,away_score
1029,178,184,2025-11-21,992.0,Gabe Arnold,15.0,301.0,Chase Kranitz,38.0,True,WMD,WMD 15 - 5,SO,Iowa,JR,Pittsburgh,Gabe Arnold,Chase Kranitz,Major Decision,False,15.0,5.0
1532,128,197,2025-12-12,992.0,Gabe Arnold,15.0,997.0,Kade Rule,48.0,True,WMD,WMD 17 - 4,SO,Iowa,FR,Chattanooga,Gabe Arnold,Kade Rule,Major Decision,False,17.0,4.0
1542,127,197,2025-12-12,992.0,Gabe Arnold,15.0,878.0,Kael Bennie,46.0,True,WDEC,WDEC 4 - 2,SO,Iowa,SO,Utah Valley,Gabe Arnold,Kael Bennie,Decision,False,4.0,2.0
3268,324,174,2026-01-16,992.0,Gabe Arnold,17.0,125.0,Levi Haines,1.0,False,LDEC,LDEC 4 - 2,SO,Iowa,SR,Penn State,Gabe Arnold,Levi Haines,Decision,False,2.0,4.0
3578,373,184,2026-01-23,222.0,Silas Allred,7.0,992.0,Gabe Arnold,13.0,False,LDEC,LDEC 4 - 1,SR,Nebraska,SO,Iowa,Silas Allred,Gabe Arnold,Decision,False,1.0,4.0
4055,413,184,2026-01-30,992.0,Gabe Arnold,13.0,156.0,Max McEnelly,3.0,False,LDEC,LDEC 4 - 1,SO,Iowa,SO,Minnesota,Gabe Arnold,Max McEnelly,Decision,False,1.0,4.0
4523,511,184,2026-02-06,449.0,Dylan Fishback,8.0,992.0,Gabe Arnold,10.0,True,WSV,WSV-1 4 - 1,JR,Ohio State,SO,Iowa,Dylan Fishback,Gabe Arnold,Decision,False,1.0,4.0
4900,500,184,2026-02-08,291.0,Ryan Boucher,128.0,992.0,Gabe Arnold,10.0,False,LTF,LTF5 20 - 4 7:00,SR,Michigan State,SO,Iowa,Ryan Boucher,Gabe Arnold,Tech Fall,False,NaN,NaN
4969,470,184,2026-02-13,992.0,Gabe Arnold,10.0,281.0,Brock Mantanona,11.0,True,WTB,WTB-1 3 - 2,SO,Iowa,RSFR,Michigan,Gabe Arnold,Brock Mantanona,Decision,False,1.0,3.0
5251,444,184,2026-02-15,232.0,James Rowley,32.0,992.0,Gabe Arnold,10.0,False,LDEC,LDEC 4 - 1,JR,Purdue,SO,Iowa,James Rowley,Gabe Arnold,Decision,False,1.0,4.0
